# JEV-LLM v3: G_SINGLE

A100 40GB; isolated v3 runtime 0.3.0, shared parent 0.2.1. Full test: 500 problems; sampled arms seeds17,29,43, greedy arms seed17 only. 

Explicit development smoke: SPLIT=dev, RUN_TAG=smoke, N_PROBLEMS=1, SEEDS=17. Use jev_llm_v3_dev_bundle.zip. No full data/seal is implied. 

Upload the explicit bundle before running; no legacy root or bundle fallback. Confirmatory execution requires APPROVAL_FILE supplied separately by the operator. All GPU phases, including smoke/pilot, share the operator's fixed 25 A100-hour accumulated ledger; the parent supplies initial authorization and post-pilot confirmation, never this notebook builder. Papermill output must be condition-bound: G_SINGLE.out.main.ipynb (or .out.smoke.ipynb). Final archive: {CONDITION}_{RUN_TAG}_{config_hash[:8]}_final.zip. No VM is allocated by this builder.

In [ ]:
# @title Install pinned stack
# Instala el stack fijado ANTES de importar torch en este kernel.
# vLLM 0.30.0 trae su propio torch/transformers; jevk5 se instala sin dependencias
# para no alterar ese stack. Si ya está instalado (re-ejecución), no hace nada.
import importlib.metadata as md, subprocess, sys, time, json
def _v(p):
    try: return md.version(p)
    except md.PackageNotFoundError: return None
t0 = time.time()
need = []
if _v("vllm") != "0.30.0": need.append("vllm==0.30.0")
for p in ("nvidia-ml-py", "papermill", "pandas", "matplotlib", "tabulate"):
    if _v(p) is None: need.append(p)
if need:
    print("instalando:", need, flush=True)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode: raise SystemExit("pip install falló (ver arriba)")
# Colab trae torchaudio/torchcodec compilados para otro CUDA; transformers los importa si
# existen y falla al mezclar versiones. No se usan en este experimento: se desinstalan.
for p in ("torchaudio", "torchcodec"):
    if _v(p) is not None:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", p])
        print("desinstalado (incompatible con el torch de vLLM):", p)
def _jevk5_commit():
    try:
        return json.loads(md.distribution("jevk5").read_text("direct_url.json") or "{}").get("vcs_info", {}).get("commit_id")
    except (md.PackageNotFoundError, ValueError):
        return None
if _v("jevk5") is None or _jevk5_commit() != "6c6522fe5462a05fdb82bceeb0e8c624c11f1517":
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                         "--force-reinstall", "git+https://github.com/allebee/jevk5@6c6522fe5462a05fdb82bceeb0e8c624c11f1517"], capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    if r.returncode: raise SystemExit("instalación de jevk5 falló")
print({p: _v(p) for p in ("vllm", "torch", "transformers", "jevk5", "nvidia-ml-py")})
print(f"instalación lista en {time.time()-t0:.0f} s")

In [ ]:
# @title Parameters
CONDITION = "G_SINGLE"
CONFIG_FILE = "config/experiment_v3.json"  # @param {type:"string"}
DATA_SUBDIR = "data/v3"  # @param {type:"string"}
ROOT = "/content/jev_llm_v3"  # @param {type:"string"}
BUNDLE_FILE = "/content/jev_llm_v3_bundle.zip"  # @param {type:"string"}
RUN_TAG = "main"  # @param {type:"string"}
SPLIT = "test"  # @param ["test", "pilot", "dev"]
N_PROBLEMS = 500  # @param {type:"integer"}
SEEDS = "17,29,43"  # @param {type:"string"}
APPROVAL_FILE = ""  # @param {type:"string"}
PILOT_REPORT_FILE = ""  # @param {type:"string"}
CONFIRMATORY_AUTHORIZED = False  # @param {type:"boolean"}
PREFLIGHT_MODE = "full"
ABORT_ON_PREFLIGHT_FAIL = True
HASH_WEIGHTS = True
REQUIRED_GPU = "A100"
WARMUP_N = 3  # @param {type:"integer"}
CHECKPOINT_EVERY = 5  # @param {type:"integer"}

In [ ]:
# @title Strict v3 bundle loader
import ast, hashlib, json, re, stat, zipfile, sys, os
from collections import Counter
from pathlib import Path, PurePosixPath
CONFIG = 'config/experiment_v3.json'
OFFICIAL_Q9_REVISION = 'c202236235762e1c871ad0ccb60c8ee5ba337b9a'
OPERATIONAL_POLICY_ID = 'jev-v3-balanced-residencies-25h-20261006'
REVIEW_POLICY_ID = 'jev-v3-source-sampling-tier-agent-review-20261004'
REVIEW_CONTRACT_SHA256 = '4ad2d4bbae07425864bcaa3fa5c99f8aa140c8feeaf902955a6bc8041c5689cb'
PUBLIC_DATA = ('test_inputs.jsonl', 'pilot_inputs.jsonl', 'dev_inputs.jsonl', 'latency_inputs.jsonl', 'schedule.json', 'latency_plan.json', 'dataset_manifest.json', 'REVIEW_CONTRACT.md', 'review_policy.json', 'ELIGIBILITY_POLICY.md', 'SHA256SUMS')
GOLD_DATA = ('test_gold.jsonl', 'pilot_gold.jsonl', 'dev_gold.jsonl')
SHARED_MODULES = ('__init__', 'algorithms', 'analysis', 'common', 'engines', 'evaluate', 'hw', 'hwprobe', 'latency_study', 'logutil', 'models', 'native_ref', 'parsing', 'preflight', 'prompts', 'runner', 'seeds', 'selector')
TOOL_SOURCES = ('tools/build_notebooks_v3.py', 'tools/make_bundle_v3.py', 'tools/colab/run_v3.py', 'tools/colab/launch_v3.sh', 'tools/colab/operator_v3.sh', 'tools/colab/session_guard.sh', 'tools/colab/session_guard.py', 'tools/colab/run_v2.py', 'tools/verify_run.py', 'tools/build_notebooks.py', 'tools/pilot_decision_v3.py')
def sha(data):
    return hashlib.sha256(data).hexdigest()

def safe_name(name):
    if (not isinstance(name, str) or not name or "\\" in name or ":" in name
            or PurePosixPath(name).is_absolute() or any(p in ("", ".", "..") for p in name.split("/"))):
        raise ValueError(f"Unsafe member/source path: {name!r}")
    return name

def input_rows(raw, count, name):
    rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
    if (len(rows) != count or any(not isinstance(r, dict) or set(r) != {"id", "problem", "language"}
            or r["language"] != "en" or any(not isinstance(v, str) or not v.strip() for v in r.values()) for r in rows)
            or len({r["id"] for r in rows}) != count):
        raise ValueError(f"Invalid blind inputs (expected {count}): {name}")
    return rows

def validate_config(cfg):
    if (cfg.get("protocol_version") != "3" or cfg.get("code_version") != "0.3.0"
            or cfg.get("implementation_profile") != "1COPY-G4-VLLM-GRAPH-V3"):
        raise ValueError("Requires exact v3 config/code/profile")
    models = cfg.get("models", {})
    if (set(models) != {"G", "J", "B", "O"} or any(not m.get("repo") or not re.fullmatch(r"[0-9a-f]{40}", m.get("revision", "")) for m in models.values())
            or models["O"].get("repo") != "Qwen/Qwen3.5-9B" or models["O"].get("revision") != OFFICIAL_Q9_REVISION):
        raise ValueError("Requires the exact pinned official Qwen3.5-9B revision and four frozen model roles")

def validate_public(raw, dev):
    """Check the actual public dataset contract and sealed blind strata."""
    metadata = json.loads(raw["data/v3/dataset_manifest.json"])
    allowed = {"dataset_version", "sealed", "status", "actual_n", "construction_seed", "latency_seed",
               "review", "input_sha256", "original_manifest_sha256", "original_seal_sha256", "development_only",
               "requested_n", "quota", "builder_sha256", "analysis_manifest_sha256", "policy_artifact_sha256",
                "source_revisions", "test_reserved_before_pilot", "latency_test_subset_only", "max_prompt_tokens", "max_abs_answer", "quota_axis"}
    if not set(metadata) <= allowed or not set(metadata.get("review", {})) <= {
            "agent_reviewed_all", "human_reviewed", "dev_agent_reviewed", "required_independent_actual_agents",
            "policy_id", "policy_sha256", "reviewed", "scope", "intrinsic_difficulty_counts", "source_sampling_tier_is_intrinsic_difficulty"}:
        raise ValueError("Private/unapproved fields in public dataset manifest")
    for name in ("builder_sha256", "analysis_manifest_sha256", "original_manifest_sha256", "original_seal_sha256"):
        if name in metadata and (not isinstance(metadata[name], str) or not re.fullmatch(r"[0-9a-f]{64}", metadata[name])):
            raise ValueError("Public provenance commitments must be SHA-256 strings")
    for source in metadata.get("source_revisions", {}).values():
        if not isinstance(source, dict) or not set(source) <= {"repo", "revision", "license"} or any(not isinstance(v, str) for v in source.values()):
            raise ValueError("Private/unapproved source revision payload")
    if ("requested_n" in metadata and metadata["requested_n"] != {"test": 500, "pilot": 50, "dev": 20, "latency": 100}
            or "quota" in metadata and metadata["quota"] != {"test": {"easy": 30, "medium": 40, "hard": 30},
                "pilot": {"easy": 3, "medium": 4, "hard": 3}, "latency": {"easy": 6, "medium": 8, "hard": 6}}):
        raise ValueError("Public quota/count contract changed")
    if dev:
        if (set(metadata) != {"dataset_version", "sealed", "development_only", "actual_n", "review", "input_sha256"}
                or set(metadata.get("review", {})) != {"agent_reviewed_all", "human_reviewed"}
                or metadata.get("sealed") is not False or metadata.get("development_only") is not True
                or metadata.get("dataset_version") != "v3-dev" or metadata.get("actual_n") != {"dev": 20}
                or metadata.get("review", {}).get("agent_reviewed_all") is not False):
            raise ValueError("Development manifest must not claim full/sealed data")
    else:
        review = metadata.get("review", {})
        if (metadata.get("sealed") is not True or metadata.get("dataset_version") != "v3"
                or metadata.get("actual_n") != {"test": 500, "pilot": 50, "dev": 20, "latency": 100}
                or review.get("agent_reviewed_all") is not True or review.get("human_reviewed") is not False
                or review.get("policy_id") != REVIEW_POLICY_ID or review.get("policy_sha256") != REVIEW_CONTRACT_SHA256
                or review["policy_sha256"] != sha(raw["data/v3/REVIEW_CONTRACT.md"])
                or metadata.get("quota_axis") != "source_sampling_tier"
                or review.get("source_sampling_tier_is_intrinsic_difficulty") is not False):
            raise ValueError("Invalid sealed public v3 dataset/review contract")
        counts = review.get("intrinsic_difficulty_counts")
        if (not isinstance(counts, dict) or set(counts) != {"test", "pilot"} or any(
                not isinstance(counts[s], dict) or set(counts[s]) != {"easy", "medium", "hard"}
                or any(type(n) is not int or n < 0 for n in counts[s].values())
                or sum(counts[s].values()) != count for s, count in (("test", 500), ("pilot", 50)))):
            raise ValueError("Public publication requires actual intrinsic difficulty counts, not source quotas")
        if ("reviewed" in review and review["reviewed"] is not False or "scope" in review and review["scope"] != ["test", "pilot"]
                or "dev_agent_reviewed" in review and review["dev_agent_reviewed"] is not False):
            raise ValueError("Public review scope must not claim human/dev review")
        policy = json.loads(raw["data/v3/review_policy.json"])
        if not isinstance(policy, dict) or set(policy) != {"policy_id", "contract", "contract_sha256", "eligibility_document_sha256", "method",
                "required_independent_reviewers", "human_review_required", "reviewed_field_means_human_review",
                "dev_reuse_not_new_agent_review", "domain_policy", "difficulty_policy", "adjudication", "eligibility", "seeds", "quota", "quota_axis", "sampling_tier", "scope_integrity"}:
            raise ValueError("Private/unapproved policy payload")
        # Match the publisher's nested public shape, not arbitrary dictionaries.
        shapes = {
            "scope_integrity": {"rule_id", "incident_ledger", "threshold", "reason", "original_verdicts_and_registrations_immutable", "replacement_reviews_cannot_reverse_disqualification"},
            "domain_policy": {"version", "applies_to", "precedence", "ratio_boundaries", "geometry_boundaries", "arithmetic", "regex_evidence"},
            "difficulty_policy": {"gsm8k", "math", "asdiv", "competition", "openstax", "disagreement"},
            "adjudication": {"required_actual_independent_agents", "original_reviews_immutable", "final_labels", "units", "gold", "vetoes"},
            "eligibility": {"language", "exact_integer_fraction_finite_decimal", "max_abs_answer", "max_rendered_tokens_each_pinned_G4_B13_O9",
                            "money_and_percent_allowed", "dollar_math_delimiters_not_currency_exclusions", "units", "non_scalar", "publisher_extraction"},
            "seeds": {"construction", "latency"}, "quota": {"test", "pilot", "latency"}}
        for key, keys in shapes.items():
            if not isinstance(policy.get(key), dict) or set(policy[key]) != keys:
                raise ValueError("Private/unapproved nested policy payload: " + key)
        for key, value in policy.items():
            if key in shapes:
                continue
            expected_type = (bool if key in {"human_review_required", "reviewed_field_means_human_review",
                "dev_reuse_not_new_agent_review"} else int if key == "required_independent_reviewers" else str)
            if type(value) is not expected_type:
                raise ValueError("Private/unapproved policy value")
        regex = policy["domain_policy"]["regex_evidence"]
        if (not isinstance(regex, dict) or set(regex) != {"counting", "number_theory", "formal_algebra", "quantities"}
                or any(not isinstance(v, str) for v in regex.values())
                or not isinstance(policy["domain_policy"]["precedence"], list)
                or any(not isinstance(v, str) for v in policy["domain_policy"]["precedence"])):
            raise ValueError("Private/unapproved nested domain policy payload")
        for key in ("domain_policy", "difficulty_policy", "adjudication", "eligibility", "seeds", "scope_integrity"):
            for name, value in policy[key].items():
                if key == "domain_policy" and name in {"regex_evidence", "precedence"}:
                    continue
                expected_type = (bool if name in {"original_reviews_immutable", "exact_integer_fraction_finite_decimal",
                    "money_and_percent_allowed", "dollar_math_delimiters_not_currency_exclusions",
                    "original_verdicts_and_registrations_immutable", "replacement_reviews_cannot_reverse_disqualification"} else
                    int if name in {"required_actual_independent_agents", "max_abs_answer", "max_rendered_tokens_each_pinned_G4_B13_O9",
                                    "construction", "latency"} else str)
                if type(value) is not expected_type:
                    raise ValueError("Private/unapproved nested policy value")
        if (policy["quota_axis"] != "source_sampling_tier" or not policy["sampling_tier"].strip()
                or policy["scope_integrity"]["rule_id"] != "assigned-input-integrity-v1"
                or policy["scope_integrity"]["incident_ledger"] != "reviews/review_scope_incidents.jsonl"
                or policy["scope_integrity"]["reason"] != "review_scope_breach"
                or not policy["scope_integrity"]["threshold"].strip()
                or policy["scope_integrity"]["original_verdicts_and_registrations_immutable"] is not True
                or policy["scope_integrity"]["replacement_reviews_cannot_reverse_disqualification"] is not True
                or policy["quota"] != {"test": {"easy": 30, "medium": 40, "hard": 30},
                "pilot": {"easy": 3, "medium": 4, "hard": 3}, "latency": {"easy": 6, "medium": 8, "hard": 6}}
                or policy["seeds"] != {"construction": 20261002, "latency": 20261003}
                or policy["adjudication"]["required_actual_independent_agents"] != 1
                or policy["adjudication"]["original_reviews_immutable"] is not True
                or policy["required_independent_reviewers"] != 2 or policy["human_review_required"] is not False
                or policy["reviewed_field_means_human_review"] is not True or policy["dev_reuse_not_new_agent_review"] is not True
                or policy["eligibility"]["language"] != "en" or policy["eligibility"]["max_abs_answer"] != 10**9
                or policy["eligibility"]["max_rendered_tokens_each_pinned_G4_B13_O9"] != 1024
                or any(policy["eligibility"][k] is not True for k in ("exact_integer_fraction_finite_decimal",
                    "money_and_percent_allowed", "dollar_math_delimiters_not_currency_exclusions"))):
            raise ValueError("Current public policy controls changed")
        if (policy.get("policy_id") != review["policy_id"] or policy.get("contract") != "REVIEW_CONTRACT.md"
                or policy.get("contract_sha256") != review["policy_sha256"]
                or policy.get("eligibility_document_sha256") != sha(raw["data/v3/ELIGIBILITY_POLICY.md"])):
            raise ValueError("Public policy identity/hash mismatch")
        policy_hashes = metadata.get("policy_artifact_sha256", {})
        if policy_hashes and (set(policy_hashes) != {"review_policy.json", "ELIGIBILITY_POLICY.md"}
                or any(h != sha(raw["data/v3/" + n]) for n, h in policy_hashes.items())):
            raise ValueError("Public policy artifact commitments changed")
        seals = {}
        for line in raw["data/v3/SHA256SUMS"].decode().splitlines():
            digest, name = line.split(maxsplit=1)
            safe_name(name)
            if name in seals or not re.fullmatch(r"[0-9a-f]{64}", digest):
                raise ValueError("Invalid public seal entry")
            seals[name] = digest
        expected = set(PUBLIC_DATA) - {"SHA256SUMS"} | set(GOLD_DATA)
        if not expected <= set(seals):
            raise ValueError("Unexpected/missing public seal entries")
        for name, digest in seals.items():
            member = "data/v3/" + name
            if member in raw and sha(raw[member]) != digest:
                raise ValueError(f"Public seal mismatch: {name}")
        test = input_rows(raw["data/v3/test_inputs.jsonl"], 500, "test")
        pilot = input_rows(raw["data/v3/pilot_inputs.jsonl"], 50, "pilot")
        dev_rows = input_rows(raw["data/v3/dev_inputs.jsonl"], 20, "dev")
        groups = [{r["id"] for r in rows} for rows in (test, pilot, dev_rows)]
        if any(groups[i] & groups[j] for i in range(3) for j in range(i)):
            raise ValueError("Overlapping public split IDs")
        schedule = json.loads(raw["data/v3/schedule.json"])
        if not set(schedule) <= {"planning_seed", "order", "pilot_order", "blocks", "pilot_blocks"}:
            raise ValueError("Private/unapproved schedule fields")
        for rows, key, blocks_key, block_count in ((test, "order", "blocks", 50), (pilot, "pilot_order", "pilot_blocks", 5)):
            order = schedule.get(key, [])
            if len(order) != len(rows) or len(set(order)) != len(order) or set(order) != {r["id"] for r in rows}:
                raise ValueError("Public schedule mismatch")
            if blocks_key in schedule and (len(schedule[blocks_key]) != block_count or any(len(b) != 10 for b in schedule[blocks_key])
                    or [pid for block in schedule[blocks_key] for pid in block] != order):
                raise ValueError("Public balanced schedule blocks mismatch")
        subset = input_rows(raw["data/v3/latency_inputs.jsonl"], 100, "latency")
        test_by_id = {r["id"]: r for r in test}
        if any(test_by_id.get(r["id"]) != r for r in subset):
            raise ValueError("Latency/test content mismatch")
        plan = json.loads(raw["data/v3/latency_plan.json"])
        if set(plan) != {"items", "dataset_version", "sealed", "selection_seed", "source_split", "split", "blocks",
                         "seeds", "repetitions", "conditions", "generation", "expected_rows", "quota_per_domain", "quota_axis", "intrinsic_difficulty_counts"}:
            raise ValueError("Missing or unexpected/private latency plan fields")
        for key, expected_value in {"dataset_version": "v3", "sealed": True, "selection_seed": 20261003, "source_split": "test",
                "split": "test", "seeds": [17, 29, 43], "repetitions": 3, "conditions": ["JFINAL", "B13_GREEDY"],
                "generation": 0, "expected_rows": 1800, "quota_per_domain": {"easy": 6, "medium": 8, "hard": 6},
                "quota_axis": "source_sampling_tier"}.items():
            if plan[key] != expected_value:
                raise ValueError("Frozen latency plan control mismatch: " + key)
        items = plan["items"]
        if len(items) != 100 or {i["item_id"] for i in items} != {r["id"] for r in subset}:
            raise ValueError("Latency plan coverage mismatch")
        for item in items:
            if (set(item) != {"item_id", "problem", "domain", "difficulty", "sampling_tier"}
                    or item["problem"] != test_by_id[item["item_id"]]["problem"]
                    or item["difficulty"] not in {"easy", "medium", "hard"}
                    or item["sampling_tier"] not in {"easy", "medium", "hard"}):
                raise ValueError("Private fields or changed content in latency plan")
        expected_order = [pid for pid in schedule["order"] if pid in {r["id"] for r in subset}]
        if [i["item_id"] for i in items] != expected_order or plan["blocks"] != [expected_order[i:i + 10] for i in range(0, 100, 10)]:
            raise ValueError("Frozen latency plan block order mismatch")
        domains = {"arithmetic", "algebra", "ratios_percentages", "number_theory", "counting_probability"}
        if {i["domain"] for i in items} != domains or any(
                Counter(i["sampling_tier"] for i in items if i["domain"] == d) != {"easy": 6, "medium": 8, "hard": 6}
                for d in domains):
            raise ValueError("Invalid blind latency source sampling strata")
        latency_counts = plan["intrinsic_difficulty_counts"]
        actual_counts = {label: sum(i["difficulty"] == label for i in items) for label in ("easy", "medium", "hard")}
        if (not isinstance(latency_counts, dict) or set(latency_counts) != set(actual_counts)
                or any(type(n) is not int or n < 0 for n in latency_counts.values()) or latency_counts != actual_counts):
            raise ValueError("Latency intrinsic difficulty counts differ from actual item labels")
        for split, rows in (("test", test), ("pilot", pilot)):
            member = f"data/v3/{split}_gold.jsonl"
            if member not in raw:
                continue
            gold = [json.loads(line) for line in raw[member].decode().splitlines() if line.strip()]
            if (len(gold) != len(rows) or [r["id"] for r in gold] != [r["id"] for r in rows]
                    or any(r.get("difficulty") not in {"easy", "medium", "hard"} for r in gold)
                    or counts[split] != {label: sum(g["difficulty"] == label for g in gold) for label in ("easy", "medium", "hard")}):
                raise ValueError("Public intrinsic difficulty counts differ from analysis gold")
            if split == "test":
                indexed = {r["id"]: r for r in gold}
                if any(any(item[k] != indexed[item["item_id"]].get(k) for k in ("domain", "difficulty", "sampling_tier")) for item in items):
                    raise ValueError("Latency reviewed labels/source tiers differ from analysis gold")
    hashes = metadata.get("input_sha256", {})
    expected = {"dev_inputs.jsonl"} if dev else {"test_inputs.jsonl", "pilot_inputs.jsonl", "dev_inputs.jsonl",
                                               "latency_inputs.jsonl", "schedule.json", "latency_plan.json"}
    if set(hashes) != expected or any(sha(raw["data/v3/" + n]) != h for n, h in hashes.items()):
        raise ValueError("Public input hashes mismatch")

def inspect_bundle(path, root=ROOT, check_local=True):
    """Validate ZIP inventory, visibility, pinned metadata and optionally local sources."""
    try:
        with zipfile.ZipFile(path) as archive:
            if sum(info.file_size for info in archive.infolist()) > 512 * 1024 * 1024:
                raise ValueError("Bundle uncompressed size exceeds 512 MiB")
            names = archive.namelist()
            if len(names) != len(set(names)):
                raise ValueError("Duplicate ZIP members")
            for info in archive.infolist():
                safe_name(info.orig_filename)
                safe_name(info.filename)
                if info.is_dir() or stat.S_ISLNK(info.external_attr >> 16):
                    raise ValueError("Directory/symlink ZIP member forbidden")
            manifest = json.loads(archive.read("BUNDLE_MANIFEST.json"))
            if (manifest.get("series") != "v3" or manifest.get("code_version") != "0.3.0"
                    or manifest.get("legacy_code_version") != "0.2.1"):
                raise ValueError("Unknown bundle series/code version")
            files = manifest["files"]
            if not isinstance(files, dict) or set(names) != set(files) | {"BUNDLE_MANIFEST.json"}:
                raise ValueError("Missing hashes or extra/missing ZIP members")
            raw = {}
            for name, digest in files.items():
                safe_name(name)
                raw[name] = archive.read(name)
                if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest) or sha(raw[name]) != digest:
                    raise ValueError(f"Bundle hash mismatch: {name}")
            dev, gold = manifest["development_only"], manifest["contains_gold"]
            if type(dev) is not bool or type(gold) is not bool or (dev and gold):
                raise ValueError("Invalid bundle visibility flags")
            allowed_data = {"dev_inputs.jsonl", "dataset_manifest.json"} if dev else set(PUBLIC_DATA) | (set(GOLD_DATA) if gold else set())
            required = {CONFIG, "tools/verify_run_v3.py", "src/jevlab/__init__.py", "src/jevlab/v3/__init__.py"}
            required |= {"src/jevlab/" + n + ".py" for n in SHARED_MODULES}
            required |= {"data/v3/" + n for n in allowed_data}
            required |= {f"src/jevlab/v3/{n}.py" for n in ("runner", "analysis", "algorithms", "latency_study", "preflight")}
            required |= {f"prompts/{n}.txt" for n in ("generador", "criterio_paso", "criterio_final")}
            if not required <= set(files):
                raise ValueError("Missing required v3 bundle members")
            for name in files:
                source = name.startswith("src/jevlab/") and name.endswith(".py") and (
                    name.count("/") == 2 or name.startswith("src/jevlab/v3/"))
                if name not in required and not source:
                    raise ValueError(f"Unapproved bundle member: {name}")
                if name.endswith("_inputs.jsonl"):
                    count = {"test": 500, "pilot": 50, "dev": 20, "latency": 100}[Path(name).stem.removesuffix("_inputs")]
                    input_rows(raw[name], count, name)
            for member, version in (("src/jevlab/__init__.py", "0.2.1"), ("src/jevlab/v3/__init__.py", "0.3.0")):
                versions = [ast.literal_eval(n.value) for n in ast.parse(raw[member]).body
                            if isinstance(n, ast.Assign) and any(isinstance(t, ast.Name) and t.id == "__version__" for t in n.targets)]
                if versions != [version]:
                    raise ValueError("Incorrect parent/v3 code version")
            mapping, sources = manifest["source_mapping"], manifest["source_sha256"]
            if not isinstance(mapping, dict) or not isinstance(sources, dict) or not set(mapping) <= set(files):
                raise ValueError("Invalid source mapping/inventory")
            if not set(TOOL_SOURCES) <= set(sources):
                raise ValueError("Missing immutable tooling source hashes")
            for name, digest in sources.items():
                safe_name(name)
                if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest):
                    raise ValueError("Invalid local source hash")
            for member, original in mapping.items():
                safe_name(original)
                if sources.get(original) != files[member]:
                    raise ValueError("Source mapping hash mismatch")
            if not required - ({"data/v3/dataset_manifest.json"} if dev else set()) <= set(mapping):
                raise ValueError("Missing source mapping")
            validate_public(raw, dev)
            if ("src/jevlab/v3/operational.py" in raw or "operational_policy_id" in manifest
                    or "operational_module_sha256" in manifest):
                operational = "src/jevlab/v3/operational.py"
                if (manifest.get("operational_policy_id") != OPERATIONAL_POLICY_ID or operational not in raw
                        or manifest.get("operational_module_sha256") != sha(raw[operational])
                        or mapping.get(operational) != operational):
                    raise ValueError("Operational policy/module source binding mismatch")
            cfg = json.loads(raw[CONFIG])
            validate_config(cfg)
            if not dev:
                review = json.loads(raw["data/v3/dataset_manifest.json"])["review"]
                if cfg.get("dataset_review_policy") != {"id": review["policy_id"], "sha256": review["policy_sha256"]}:
                    raise ValueError("Bundle/config review policy pin mismatch")
            for key, value in (("config_sha256", sha(raw[CONFIG])), ("models", cfg["models"]),
                               ("implementation_profile", cfg["implementation_profile"]),
                               ("jevk5_runtime", cfg["jevk5_runtime"]), ("software_pins", cfg["software_pins"])):
                if manifest.get(key) != value:
                    raise ValueError(f"Bundle/config pin mismatch: {key}")
            if cfg.get("code_version") != "0.3.0" or cfg.get("protocol_version") != "3":
                raise ValueError("Invalid v3 configuration")
            if check_local:
                payload, expected = _payload(Path(root).resolve(), gold, dev)
                if manifest != expected or raw != payload:
                    raise ValueError("Bundle differs from local source integrity/actual public artifacts")
            return manifest
    except (OSError, KeyError, TypeError, UnicodeError, SyntaxError, json.JSONDecodeError, zipfile.BadZipFile) as exc:
        raise ValueError(f"Invalid v3 bundle: {exc}") from exc

ROOT = Path(ROOT)
ZIP = Path(BUNDLE_FILE)
if not ZIP.is_file():
    raise ValueError(f"Missing explicit v3 bundle: {ZIP}; upload it before running")
man = inspect_bundle(ZIP, check_local=False)
if man["contains_gold"] is not False:
    raise ValueError("Wrong bundle gold visibility for this notebook")
if man["development_only"] and SPLIT != "dev":
    raise ValueError("Development bundle cannot run test/pilot")
if not False and SPLIT == "dev" and (RUN_TAG != "smoke" or SEEDS != "17"):
    raise ValueError("Explicit dev smoke requires RUN_TAG=smoke and SEEDS=17")
marker = ROOT / "BUNDLE_MANIFEST.json"
if ROOT.is_symlink():
    raise ValueError("Symlink ROOT forbidden")
if ROOT.exists() and any(ROOT.iterdir()):
    if not marker.is_file() or json.loads(marker.read_text(encoding="utf-8")) != man:
        raise ValueError("Stale/foreign root: use an empty isolated v3 ROOT, never legacy fallback")
    for name, digest in man["files"].items():
        path = ROOT / name
        if (path.is_symlink() or not path.resolve().is_relative_to(ROOT.resolve())
                or any(p.is_symlink() for p in path.parents) or not path.is_file() or sha(path.read_bytes()) != digest):
            raise ValueError(f"Stale root hash mismatch: {name}")
    for directory in ("src", "data", "config", "prompts", "tools"):
        for path in (ROOT / directory).rglob("*"):
            name = path.relative_to(ROOT).as_posix()
            if path.is_symlink() or (path.is_file() and "__pycache__" not in path.parts and name not in man["files"]):
                raise ValueError(f"Extra/stale root member: {name}")
else:
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP) as archive:
        for name in [*man["files"], "BUNDLE_MANIFEST.json"]:
            path = ROOT / name
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(name))
sys.path.insert(0, str(ROOT / "src"))
import jevlab, jevlab.v3
if jevlab.__version__ != "0.2.1" or jevlab.v3.__version__ != "0.3.0":
    raise ValueError("Wrong parent/v3 runtime versions")
for module in (jevlab, jevlab.v3):
    if not Path(module.__file__).resolve().is_relative_to((ROOT / "src").resolve()):
        raise ValueError("Previously imported foreign runtime; restart kernel")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")

In [ ]:
# @title Validate run and approval
if CONFIG_FILE != "config/experiment_v3.json" or DATA_SUBDIR != "data/v3":
    raise ValueError("Only isolated v3 config/data are accepted")
if SPLIT == "test":
    if CONFIRMATORY_AUTHORIZED is not True or not APPROVAL_FILE or not PILOT_REPORT_FILE:
        raise ValueError("Explicit authorization, user budget sentinel and actual pilot GO report are required")
    approval = json.loads(Path(APPROVAL_FILE).read_text(encoding="utf-8"))
    pilot_raw = Path(PILOT_REPORT_FILE).read_bytes()
    pilot = json.loads(pilot_raw)
    import math
    from datetime import datetime, timezone
    scopes = {"test_and_latency"} if CONDITION == "ANALYSIS" else {"latency", "test_and_latency"} if CONDITION == "LATENCY" else {"test", "test_and_latency"}
    if (approval.get("approved") is not True or approval.get("scope") not in scopes
            or approval.get("config_sha256") != man["config_sha256"]
            or approval.get("pilot_report_sha256") != hashlib.sha256(pilot_raw).hexdigest()
            or not isinstance(approval.get("approved_by"), str) or not approval["approved_by"].strip() or not approval.get("approved_utc")
            or type(approval.get("max_gpu_hours")) not in (int, float)
            or not math.isfinite(approval["max_gpu_hours"]) or not 0 < approval["max_gpu_hours"] <= 25):
        raise ValueError("Invalid budget approval sentinel")
    arms = {"G_SINGLE", "G_GREEDY", "JFINAL", "B13", "B13_GREEDY", "Q9_GREEDY"}
    entries = pilot.get("conditions", {})
    aggregate = pilot.get("aggregate", {})
    if (pilot.get("protocol_version") != "3" or pilot.get("code_version") != "0.3.0"
            or pilot.get("go") is not True or pilot.get("decision") != "go" or pilot.get("errors") != []
            or pilot.get("budget_approval") is not False or set(entries) != arms
            or aggregate.get("complete") is not True or aggregate.get("denominator") != 600
            or aggregate.get("observed_cases") != 600 or not 0 <= aggregate.get("timeout_infra_rate", 1) < .05):
        raise ValueError("Actual complete technical pilot GO required, not a quality-win rule")
    for arm, entry in entries.items():
        count = 150 if arm in {"G_SINGLE", "JFINAL", "B13"} else 50
        if (entry.get("verified") is not True or entry.get("denominator") != count
                or entry.get("verification", {}).get("ok") is not True
                or entry.get("verification", {}).get("condition") != arm
                or entry.get("verification", {}).get("records") != count
                or entry.get("rates", {}).get("denominator") != count
                or not 0 <= entry["rates"].get("timeout_infra_rate", 1) < .05):
            raise ValueError("Incomplete/failed technical pilot arm")
    approved_at = datetime.fromisoformat(approval["approved_utc"].replace("Z", "+00:00"))
    if approved_at.tzinfo is None or approved_at > datetime.now(timezone.utc):
        raise ValueError("Approval must precede execution")
if SPLIT == "dev" and N_PROBLEMS != 1:
    raise ValueError("Quality smoke requires N_PROBLEMS=1")
if SPLIT == "test" and N_PROBLEMS != 500:
    raise ValueError("Full quality requires N_PROBLEMS=500")
if SPLIT != "dev" and SEEDS != ("17" if CONDITION.endswith("GREEDY") else "17,29,43"):
    raise ValueError("Full quality seed schedule must match the physical condition")

In [ ]:
# @title Create v3 runtime
from jevlab.v3.runner import Experiment
exp = Experiment(**dict(CONDITION=CONDITION, CONFIG_FILE=CONFIG_FILE, DATA_SUBDIR=DATA_SUBDIR, ROOT=ROOT, RUN_TAG=RUN_TAG, SPLIT=SPLIT, N_PROBLEMS=N_PROBLEMS, SEEDS=SEEDS, APPROVAL_FILE=APPROVAL_FILE, PILOT_REPORT_FILE=PILOT_REPORT_FILE, CONFIRMATORY_AUTHORIZED=CONFIRMATORY_AUTHORIZED, PREFLIGHT_MODE=PREFLIGHT_MODE, ABORT_ON_PREFLIGHT_FAIL=ABORT_ON_PREFLIGHT_FAIL, HASH_WEIGHTS=HASH_WEIGHTS, REQUIRED_GPU=REQUIRED_GPU, WARMUP_N=WARMUP_N, CHECKPOINT_EVERY=CHECKPOINT_EVERY))
print(exp.dir)

In [ ]:
# @title Run and release GPU
try:
    exp.capture_environment()
    exp.prepare_models()
    exp.preflight_before_engines()
    exp.start_engines()
    exp.preflight_after_engines()
    exp.warmup()
    exp.run()
    final_archive = exp.finalize()
    print("Final archive:", final_archive)
finally:
    exp.shutdown()